# M5 Walmart Demand Forecasting — Phase 1: Exploratory Data Analysis

**Store:** CA_1 | **Granularity:** Daily aggregated sales (all items) | **Date range:** Jan 2011 – May 2016


---
## 1. Business Context

Walmart's **CA_1 store** in California stocks approximately **3,049 unique SKUs** across food, household,
and hobbies departments. Sales data spans **January 29, 2011 to June 19, 2016** — 1,913 days of daily
sales history. This EDA focuses on the **daily aggregated demand signal** (all items summed) to
understand time series properties before modelling.

### Why demand forecasting matters

| Problem | Business Cost |
|---|---|
| **Overstock** | Holding cost ≈ 20–30% of item value/year; forced markdowns erode margins |
| **Stockout** | Lost revenue + customer churn; ~4% annual sales loss industry-wide |
| **Labour planning** | Inaccurate forecasts → overstaffing or understaffing at checkout/stocking |
| **Supply chain** | Poor demand signal inflates supplier order variance (bullwhip effect) |

### Questions this EDA answers

1. Is there a long-term upward/downward **trend** in CA_1 sales?
2. What are the dominant **seasonality patterns** — weekly, monthly, annual?
3. Do **SNAP food-stamp disbursement days** create statistically meaningful demand spikes?
4. Which **calendar events** drive the largest sales anomalies?
5. Is the series **stationary**? What order of differencing (d) is needed for SARIMA?
6. What **feature engineering** is implied by the ACF/PACF structure?


---
## 2. Data Loading & Scoping (CA_1 Store Only)


In [ ]:
import sys
import os

# Add src/ to path so 'from data_loader import ...' works directly
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Plot aesthetics ──────────────────────────────────────────────────────────
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')

os.makedirs('../outputs/plots', exist_ok=True)

# ── Load data ────────────────────────────────────────────────────────────────
from data_loader import load_m5_data

df = load_m5_data()

print(f'Shape      : {df.shape}')
print(f'Date range : {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Columns    : {list(df.columns)}')
print()
print('Basic statistics:')
print(df['sales'].describe().round(1))

In [ ]:
df.head()

---
## 3. Time Series Overview


In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(
    df['date'], df['sales'],
    linewidth=0.6, alpha=0.55, color='steelblue', label='Daily Sales'
)

rolling_30 = df['sales'].rolling(30, center=True).mean()
ax.plot(
    df['date'], rolling_30,
    linewidth=2.2, color='crimson', label='30-day Rolling Mean'
)

ax.set_title('CA_1 Daily Sales — Jan 2011 to May 2016', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Units Sold (all items)')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend()

plt.tight_layout()
fig.savefig('../outputs/plots/01_timeseries_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 01_timeseries_overview.png')

---
## 4. Trend Analysis


In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Set date as index with daily frequency
sales_series = df.set_index('date')['sales'].asfreq('D')

# Additive decomposition — period=365 isolates annual seasonality
decomp = seasonal_decompose(sales_series, model='additive', period=365)

fig = decomp.plot()
fig.set_size_inches(14, 10)
plt.suptitle('Seasonal Decomposition — CA_1 Daily Sales', y=1.01, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/plots/02_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 02_decomposition.png')

### Trend Interpretation

The **trend component** (second panel) isolates the long-run direction of CA_1 sales after removing
seasonality and residual noise.

- **2011–2013:** Gradual upward growth, reflecting increasing customer base and store maturation.
- **2013–2015:** Relatively stable plateau — the store has reached a steady-state sales level.
- **2015–2016:** Mild uptick, potentially driven by new product categories or improved in-store availability.

The **seasonal component** (third panel) captures a repeating annual pattern — note the consistent
peaks and troughs at the same calendar positions each year (holiday season, summer lull).

The **residual component** (fourth panel) represents unexplained variation — single-day spikes
here correspond to anomalous events (Super Bowl, storm days, etc.) that we'll isolate in Section 6.

> **Modelling implication:** A non-constant trend means the series is not stationary in the mean.
> SARIMA will require at least **d=1** (first differencing). Prophet handles trend natively via
> piecewise linear segments. LSTM captures trend through the hidden state over a long look-back window.


---
## 5. Seasonality Analysis (Weekly / Monthly / Yearly)


In [ ]:
# ── Weekly seasonality ───────────────────────────────────────────────────────
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
weekly_avg = df.groupby('day_of_week')['sales'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#4c72b0'] * 5 + ['#dd8452', '#dd8452']   # orange = weekends
bars = ax.bar(
    weekly_avg['day_of_week'], weekly_avg['sales'],
    color=colors, edgecolor='white', linewidth=0.5
)
ax.set_xticks(range(7))
ax.set_xticklabels(day_labels)
ax.set_title('Average Sales by Day of Week — CA_1', fontsize=13, fontweight='bold')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Average Daily Sales')

for bar, val in zip(bars, weekly_avg['sales']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 25,
        f'{val:,.0f}',
        ha='center', va='bottom', fontsize=9
    )

ax.legend(
    handles=[
        plt.Rectangle((0,0),1,1, color='#4c72b0', label='Weekday'),
        plt.Rectangle((0,0),1,1, color='#dd8452', label='Weekend')
    ]
)
plt.tight_layout()
fig.savefig('../outputs/plots/03_weekly_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()

# Quantify
weekend_avg = df[df['is_weekend']]['sales'].mean()
weekday_avg = df[~df['is_weekend']]['sales'].mean()
print(f'Weekday avg    : {weekday_avg:,.0f} units/day')
print(f'Weekend avg    : {weekend_avg:,.0f} units/day')
print(f'Weekend uplift : +{((weekend_avg / weekday_avg - 1) * 100):.1f}%')

In [ ]:
# ── Monthly seasonality ──────────────────────────────────────────────────────
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_avg = df.groupby('month')['sales'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(
    monthly_avg['month'], monthly_avg['sales'],
    color='seagreen', edgecolor='white', linewidth=0.5
)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.set_title('Average Sales by Month — CA_1', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Average Daily Sales')

for _, row in monthly_avg.iterrows():
    ax.text(
        row['month'], row['sales'] + 20,
        f"{row['sales']:,.0f}",
        ha='center', va='bottom', fontsize=8
    )

plt.tight_layout()
fig.savefig('../outputs/plots/04_monthly_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()

peak_idx   = monthly_avg['sales'].idxmax()
trough_idx = monthly_avg['sales'].idxmin()
peak_month   = month_labels[monthly_avg.loc[peak_idx,   'month'] - 1]
trough_month = month_labels[monthly_avg.loc[trough_idx, 'month'] - 1]
print(f'Peak month   : {peak_month}  ({monthly_avg.loc[peak_idx,   "sales"]:,.0f} avg units/day)')
print(f'Trough month : {trough_month}  ({monthly_avg.loc[trough_idx, "sales"]:,.0f} avg units/day)')
print(f'Seasonal swing: {((monthly_avg["sales"].max() / monthly_avg["sales"].min() - 1)*100):.1f}%')

In [ ]:
# ── SNAP day impact ──────────────────────────────────────────────────────────
snap_stats = (
    df.groupby('snap_day')['sales']
    .agg(['mean', 'median', 'std', 'count'])
    .reset_index()
)
snap_stats['snap_day'] = snap_stats['snap_day'].map({False: 'Non-SNAP Day', True: 'SNAP Day'})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart — mean
colors = ['#4c72b0', '#dd8452']
bars = axes[0].bar(
    snap_stats['snap_day'], snap_stats['mean'],
    color=colors, edgecolor='white', linewidth=0.5, width=0.5
)
for bar, val in zip(bars, snap_stats['mean']):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 25,
        f'{val:,.0f}',
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )
axes[0].set_title('Sales Impact of SNAP Days — CA_1', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Average Daily Sales')

# Box plot — distribution
snap_false = df[~df['snap_day']]['sales']
snap_true  = df[ df['snap_day']]['sales']
bp = axes[1].boxplot(
    [snap_false, snap_true],
    labels=['Non-SNAP Day', 'SNAP Day'],
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2)
)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Sales Distribution — SNAP vs Non-SNAP', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Daily Sales')

plt.tight_layout()
fig.savefig('../outputs/plots/05_snap_impact.png', dpi=150, bbox_inches='tight')
plt.show()

snap_mean    = snap_stats.loc[snap_stats['snap_day']=='SNAP Day',    'mean'].values[0]
nonsnap_mean = snap_stats.loc[snap_stats['snap_day']=='Non-SNAP Day','mean'].values[0]
snap_lift    = (snap_mean / nonsnap_mean - 1) * 100
print(f'Non-SNAP day avg : {nonsnap_mean:,.0f} units')
print(f'SNAP day avg     : {snap_mean:,.0f} units')
print(f'SNAP lift        : +{snap_lift:.1f}%')

---
## 6. Anomaly Detection & Business Explanation


In [ ]:
# ── Z-score anomaly detection ────────────────────────────────────────────────
df_z = df.copy()
mu, sigma = df_z['sales'].mean(), df_z['sales'].std()
df_z['z_score'] = (df_z['sales'] - mu) / sigma

THRESHOLD = 2.5
anomalies = df_z[df_z['z_score'].abs() > THRESHOLD].copy()
anomalies['abs_z'] = anomalies['z_score'].abs()

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(df_z['date'], df_z['sales'],
        linewidth=0.6, alpha=0.5, color='steelblue', label='Daily Sales')
ax.axhline(mu,           color='gray',   linestyle='--', linewidth=0.8, alpha=0.7, label='Mean')
ax.axhline(mu + THRESHOLD*sigma, color='orange', linestyle=':', linewidth=1,   alpha=0.8, label=f'+{THRESHOLD}σ')
ax.axhline(mu - THRESHOLD*sigma, color='orange', linestyle=':', linewidth=1,   alpha=0.8, label=f'-{THRESHOLD}σ')
ax.scatter(anomalies['date'], anomalies['sales'],
           color='crimson', zorder=5, s=45, label=f'Anomaly (|z| > {THRESHOLD})')

ax.set_title(
    f'Anomaly Detection — CA_1 Daily Sales  (Z-score threshold = {THRESHOLD}σ)',
    fontsize=13, fontweight='bold'
)
ax.set_xlabel('Date')
ax.set_ylabel('Units Sold')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend(ncol=2)

plt.tight_layout()
fig.savefig('../outputs/plots/06_anomalies.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Top 15 anomaly table ─────────────────────────────────────────────────────
top15 = anomalies.nlargest(15, 'abs_z')[['date','sales','z_score','event_name_1']].copy()
top15['event_name_1'] = top15['event_name_1'].fillna('— (no event tag)')
top15['z_score'] = top15['z_score'].round(2)
top15['date'] = top15['date'].dt.strftime('%Y-%m-%d')
top15 = top15.rename(columns={'event_name_1': 'Calendar Event'})

print(f'Total anomaly days detected: {len(anomalies)}  (|z| > {THRESHOLD})')
print()
print('Top 15 Anomalies by |Z-score|:')
print('-' * 65)
print(top15.to_string(index=False))

### Business Explanation of Top Anomalies

| Driver | Sales Effect | Business Explanation |
|---|---|---|
| **Super Bowl Sunday** | Large spike | Party foods, snacks, beverages — one of Walmart's highest single-day grocery events |
| **Pre-Christmas (Dec 22–24)** | Large spike | Last-minute shopping surge; gifts, food, party supplies |
| **Christmas Day (Dec 25)** | Sharp drop | Reduced store hours or closure; customers stay home |
| **Thanksgiving Day** | Drop | Store closed or limited hours; stock-up occurs Nov 22–23 |
| **New Year's Day** | Drop | Reduced hours + post-holiday spending lull |
| **Independence Day (Jul 4)** | Spike | BBQ supplies, beverages, seasonal items |
| **SNAP week (month start)** | Moderate spike | California SNAP (food stamps) disbursed at start of month — concentrated grocery spend |

> **Modelling implication:** Events like Super Bowl and pre-Christmas are **not captured by a simple
> calendar pattern** — they require explicit event regressors. Prophet handles this natively through
> its `holidays` dataframe. SARIMA requires manual dummy variables. LSTM can learn event effects
> if the look-back window spans multiple years, but explicit lag features are still beneficial.


---
## 7. Stationarity Test (ADF)


In [ ]:
from statsmodels.tsa.stattools import adfuller

# ── ADF Test — raw series ────────────────────────────────────────────────────
adf = adfuller(df['sales'])

print('=' * 55)
print('  Augmented Dickey-Fuller Stationarity Test')
print('  H0: The series has a unit root (non-stationary)')
print('=' * 55)
print(f'  ADF Statistic  : {adf[0]:.4f}')
print(f'  p-value        : {adf[1]:.4f}')
print(f'  Lags used      : {adf[2]}')
print(f'  Observations   : {adf[3]}')
print('  Critical values:')
for level, val in adf[4].items():
    flag = '  <-- ADF statistic' if adf[0] < val else ''
    print(f'    {level:>5}: {val:.4f}{flag}')
print('=' * 55)
if adf[1] < 0.05:
    print('  CONCLUSION: Series IS stationary (p < 0.05 -- reject H0)')
else:
    print('  CONCLUSION: Series is NOT stationary -- differencing required')
print('=' * 55)

# ── ADF Test — first-differenced series ─────────────────────────────────────
print()
adf_diff = adfuller(df['sales'].diff().dropna())
print('ADF on first-differenced series:')
print(f'  p-value: {adf_diff[1]:.4f}  |  ', end='')
if adf_diff[1] < 0.05:
    print('Stationary after d=1 differencing -- use d=1 in SARIMA')
else:
    print('Still non-stationary -- consider d=2')

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

plot_acf(
    df['sales'], lags=40, ax=axes[0],
    color='steelblue', zero=False
)
axes[0].set_title('Autocorrelation Function (ACF) — CA_1 Daily Sales', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Lag (days)')

plot_pacf(
    df['sales'], lags=40, ax=axes[1],
    color='crimson', zero=False, method='ywm'
)
axes[1].set_title('Partial Autocorrelation Function (PACF) — CA_1 Daily Sales', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Lag (days)')

plt.tight_layout()
fig.savefig('../outputs/plots/07_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 07_acf_pacf.png')

### ADF Test & ACF/PACF Interpretation

**ADF Result interpretation:**
- **p < 0.05**: Null hypothesis of a unit root is rejected — the series is stationary in the mean.
  SARIMA order **d = 0 or 1** may be sufficient.
- **p ≥ 0.05**: A unit root is present — first differencing (d = 1) will be applied. Typically,
  daily retail sales become stationary after d = 1.

**ACF Interpretation:**
- **Slow geometric decay** across many lags → confirms non-stationarity / strong trend →
  differencing needed before fitting ARIMA.
- **Spike at lag 7** (and multiples 14, 21, 28 …) → strong **weekly seasonality** →
  SARIMA seasonal term with **m = 7**.
- **Gradual elevation around lag 30–31** → monthly pattern captured by Prophet's Fourier terms.

**PACF Interpretation:**
- **Significant spike at lag 1, sharp cut-off** → suggests **AR(1)** for the non-seasonal component.
- **Spike at lag 7** → confirms **seasonal AR(1)** with m = 7.
- Pattern consistent with SARIMA**(1,1,1)(1,1,1,7)** as a starting grid-search point.

**Model Selection Implications:**

| Model | How it uses this information |
|---|---|
| **SARIMA(p,d,q)(P,D,Q,7)** | d from ADF result; p,q from PACF/ACF cut-off; seasonal m=7 from ACF spikes at 7, 14, 21 |
| **Prophet** | Handles trend + weekly/yearly seasonality natively; no manual stationarity or differencing needed |
| **LSTM** | Stationarity less critical; lag features at 1, 7, 28 encode the ACF structure explicitly |


---
## 8. Key EDA Findings Summary

### Five Key Findings

- **Trend:** CA_1 shows a gradual upward trend from 2011–2013, followed by a stable plateau through
  2015, with a mild uptick heading into 2016. The trend is non-stationary in the mean; first
  differencing (d=1) is expected to achieve stationarity for SARIMA.

- **Dominant seasonality:** Weekly seasonality is the strongest pattern — Saturday and Sunday
  generate meaningfully higher sales than weekdays (quantified above). A secondary annual pattern
  is visible, with Q4 (Oct–Dec) elevated by holiday demand and a mid-summer softness in July–August.

- **SNAP day effect:** SNAP disbursement days in California produce a consistent and measurable
  lift in daily sales (quantified above). Since ~10 days per month qualify as SNAP days in CA,
  this is a recurring, predictable demand driver — it should be included as a binary feature in
  all three models.

- **Top anomaly events:** Super Bowl Sunday and pre-Christmas (Dec 22–24) generate the largest
  positive sales spikes. Christmas Day and Thanksgiving Day show sharp dips (store closures /
  reduced hours). These effects are not captured by time patterns alone — explicit calendar
  regressors are required.

- **Stationarity:** The ADF test result (see above) guides SARIMA's differencing order. The ACF
  shows significant spikes at lag 7 (and multiples), confirming dominant weekly seasonality.
  The PACF suggests AR(1) for the non-seasonal component, pointing toward
  SARIMA(1,d,1)(1,1,1,7) as the starting candidate.

---

### Connecting EDA to Model Selection

The time series properties discovered here directly inform our three-model strategy in Phase 2:

**SARIMA** is appropriate because the series shows a clear autoregressive structure (PACF spike at
lag 1), strong weekly seasonality (ACF spikes at lags 7, 14, 21), and becomes stationary after
differencing. Its explicit parameterisation makes it interpretable and a strong statistical baseline
for benchmarking.

**Prophet** is chosen for its native handling of multiple seasonalities (weekly + yearly) and its
built-in support for holiday/event regressors — exactly the Super Bowl and Christmas effects we
identified. It also handles trend changepoints automatically, without requiring manual differencing.

**LSTM** is included because the non-linear, multi-scale temporal dependencies (daily noise, weekly
cycles, annual peaks, event spikes) exceed what linear models can efficiently represent. With a
28-day look-back window, the network learns the interaction between SNAP days, weekend effects,
and trend simultaneously — without requiring explicit stationarity preprocessing.

> **Next step:** → `notebooks/02_models.ipynb` — train all three models on the same CA_1 training
> set, generate 28-day forecasts, and compare RMSE / MAPE / SMAPE with MLflow tracking.
